In [1]:
def load_system(path: pathlib.Path) -> tuple[list[list[float]], list[float]]:
    A = []
    B = []

    index = {'x': 0, 'y': 1, 'z': 2}
    
    with open(path, 'r') as file:
        for line in file:
            coefs = [0.0, 0.0, 0.0]
            coef = ''
            
            for c in line:
                match c:
                    case ' ' | '+' | '=':
                        continue

                    case 'x' | 'y' | 'z':
                        if coef == '-':
                            coefs[index[c]] = -1.0
                        elif coef:
                            coefs[index[c]] = float(coef)
                        else:
                            coefs[index[c]] = 1.0
                            
                        coef = ''

                    case _:
                        coef += c
                        
            A.append(coefs)
            B.append(float(coef))
                        
    return A, B

In [2]:
def determinant(matrix: list[list[float]]) -> float:
    if len(matrix[0]) == 3:
        a11, a12, a13 = matrix[0]
        a21, a22, a23 = matrix[1]
        a31, a32, a33 = matrix[2]
    
        return (a11 * (a22*a33 - a23*a32)
                - a12 * (a21*a33 - a23*a31)
                + a13 * (a21*a32 - a22*a31)
               )
    else:
        a11, a12 = matrix[0]
        a21, a22 = matrix[1]
        return (a11 * a22 - a21 * a12) 

In [3]:
def trace(matrix: list[list[float]]) -> float:
    return matrix[0][0] + matrix[1][1] + matrix[2][2]

In [4]:
def norm(vector: list[float]) -> float:
    return sum([b**2 for b in B])**0.5

In [5]:
def transpose(matrix: list[list[float]]) -> list[list[float]]:
    return [list(row) for row in zip(*matrix)]

In [6]:
def multiply(matrix: list[list[float]], vector: list[float]) -> list[float]:
    result = []

    for row in matrix:
        s = 0
        for a, b in zip(row, vector):
            s += a * b
        result.append(s)

    return result

In [7]:
def solve_cramer(matrix: list[list[float]], vector: list[float]) -> list[float]:
    result = []
    det_A = determinant(matrix)

    if not det_A:
        raise ValueError("Matrix is singular")
    
    for j in range(3):
        matrix_c = [row.copy() for row in matrix]

        for i in range(3):
            matrix_c[i][j] = vector[i]

        result.append(determinant(matrix_c)/det_A)

    return result

In [8]:
def minor(matrix: list[list[float]], i: int, j: int) -> list[list[float]]:
    return [[matrix[x][y] for y in range(3) if y != j] for x in range(3) if x != i]

In [9]:
def cofactor(matrix: list[list[float]]) -> list[list[float]]:
    return [[(-1)**(i+j) * determinant(minor(matrix, i, j)) for j in range(3)] for i in range(3)]

In [10]:
def adjoint(matrix: list[list[float]]) -> list[list[float]]:
    return transpose(matrix)

In [11]:
def solve(matrix: list[list[float]], vector: list[float]) -> list[float]:
    det_A = determinant(matrix)
    if not det_A:
        raise ValueError("Matrix is singular")
    
    adjoint_matrix = adjoint(cofactor(matrix))
    inverse = [[1/det_A * adjoint_matrix[i][j] for j in range(3)] for i in range(3)]

    return multiply(inverse, vector)

In [13]:
A, B = load_system("system.txt")

print(f"{A=} {B=}\n")
print(f"{determinant(A)=}\n")
print(f"{trace(A)=}\n")
print(f"{norm(B)=}\n")
print(f"{transpose(A)=}\n")
print(f"{multiply(A, B)=}\n")
print(f"{solve_cramer(A, B)=}\n")
print(f"{solve(A, B)=}")

A=[[2.5, 0.0, -1.0], [1.0, -1.0, 4.0], [3.0, 1.3, 2.0]] B=[5.0, -6.0, 7.0]

determinant(A)=-22.3

trace(A)=3.5

norm(B)=10.488088481701515

transpose(A)=[[2.5, 1.0, 3.0], [0.0, -1.0, 1.3], [-1.0, 4.0, 2.0]]

multiply(A, B)=[5.5, 39.0, 21.2]

solve_cramer(A, B)=[1.57847533632287, 3.3632286995515694, -1.053811659192825]

solve(A, B)=[1.57847533632287, 3.3632286995515694, -1.0538116591928253]
